# GP-prior pRF fits on NPCr — results

Loads the output of `neural_priors/encoding_model/fit_gp_prior.py` and
compares three fitting regimes head-to-head — across **two BOLD pipelines**:
unsmoothed (the GP-prior's home turf) and smoothed (traditional baseline).

Fitting regimes:

- **classical** — `ParameterFitter`, SSQ loss
- **ml** — `BayesianParameterFitter(priors={})`: same MAP loop, no GP prior
- **bayes** — `BayesianParameterFitter` with GP priors on mu/sd/amplitude/baseline

Three metrics per (subject, range, method, pipeline):

1. **cvR²** of the encoding-model predictions on held-out trials
2. **Decoding MAE** of posterior-mean numerosity from FDR-selected voxels
3. **Number of FDR-significant voxels** the mixture surfaces

Key 2×2 contrast: does `bayes(unsmoothed)` match or beat `classical(smoothed)`?
If yes, the GP prior is a viable replacement for traditional spatial BOLD
smoothing — without erasing fine-grained spatial structure.


## Setup

In [ ]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
    'figure.titlesize': 17,
})
sns.set_style('whitegrid')

BIDS = Path('/data/ds-neuralpriors')
DERIV = BIDS / 'derivatives' / 'encoding_models'
PIPELINES = {  # display name → derivative subdir
    'unsmoothed': DERIV / 'gp_prior_roi-NPCr',
    'smoothed':   DERIV / 'gp_prior_roi-NPCr.smoothed',
}
ROI = 'NPCr'
METHOD_ORDER = ['classical', 'ml', 'bayes']
METHOD_COLORS = {'classical': '#777', 'ml': '#1f77b4', 'bayes': '#d62728'}
PIPELINE_ORDER = ['unsmoothed', 'smoothed']
RANGES = ['narrow', 'wide']

## Load all TSVs

Each subject writes per-range TSVs into the pipeline's derivative dir. We
glob across both unsmoothed and smoothed dirs and stack with a `pipeline`
column, so every downstream plot can facet on it.

In [ ]:
def _concat_tsv_across_pipelines(pattern):
    """Read every per-subject TSV matching ``pattern`` in any of the
    PIPELINES dirs and stack with ``subject`` + ``pipeline`` columns."""
    frames = []
    for pipeline, root in PIPELINES.items():
        for path in sorted(root.glob(pattern)):
            subj = path.name.split('_')[0].replace('sub-', '')
            df = pd.read_csv(path, sep='\t')
            df.insert(0, 'subject', subj)
            df.insert(1, 'pipeline', pipeline)
            frames.append(df)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

cvr2     = _concat_tsv_across_pipelines('sub-*/func/sub-*_desc-cvr2.tsv')
decoding = _concat_tsv_across_pipelines('sub-*/func/sub-*_desc-decoding.tsv')
trials   = _concat_tsv_across_pipelines('sub-*/func/sub-*_desc-decoded_trials.tsv')
hyperp   = _concat_tsv_across_pipelines('sub-*/func/sub-*_desc-hyperpars.tsv')

if len(cvr2):
    coverage = (cvr2.groupby('pipeline')['subject']
                .nunique().rename('n_subjects'))
    print('Subjects with cvR² output per pipeline:')
    print(coverage.to_string())
print(f'shapes: cvr2={cvr2.shape}, decoding={decoding.shape}, '
      f'trials={trials.shape}, hyperp={hyperp.shape}')

## cvR² head-to-head

Median cvR² across voxels & folds per (subject, range, method, pipeline).
Rows = BOLD pipeline, cols = stimulus range, x = fitting regime.

In [ ]:
cvr2_summary = (cvr2
                .groupby(['subject', 'pipeline', 'stim_range', 'method'])['cvr2']
                .median()
                .reset_index())

g = sns.catplot(
    data=cvr2_summary,
    x='method', y='cvr2',
    row='pipeline', col='stim_range',
    kind='box', order=METHOD_ORDER, row_order=PIPELINE_ORDER,
    palette=METHOD_COLORS, height=3.6, aspect=1.1)
g.set_axis_labels('Method', 'Median cvR² across voxels & folds')
g.set_titles('Pipeline = {row_name} | Range = {col_name}')
g.fig.suptitle('cvR² per subject — 2×2 by pipeline × range', y=1.02)
plt.show()

**Key contrast**: each subject's `bayes(unsmoothed)` − `classical(smoothed)`.
If the GP-prior on raw data matches or beats traditional smoothing, this
should hover ≥ 0.

In [ ]:
# Wide: (subject, range) × (pipeline, method) → cvR²
wide = cvr2_summary.pivot_table(
    index=['subject', 'stim_range'],
    columns=['pipeline', 'method'],
    values='cvr2').reset_index()

def _safe_get(df, pipeline, method):
    try:
        return df[(pipeline, method)]
    except KeyError:
        return pd.Series(np.nan, index=df.index)

contrasts = pd.DataFrame({
    'subject':    wide['subject'],
    'stim_range': wide['stim_range'],
    'bayes_unsm_minus_classical_unsm': (
        _safe_get(wide, 'unsmoothed', 'bayes')
        - _safe_get(wide, 'unsmoothed', 'classical')),
    'classical_sm_minus_classical_unsm': (
        _safe_get(wide, 'smoothed', 'classical')
        - _safe_get(wide, 'unsmoothed', 'classical')),
    'bayes_unsm_minus_classical_sm': (
        _safe_get(wide, 'unsmoothed', 'bayes')
        - _safe_get(wide, 'smoothed', 'classical')),
})
contrasts_long = contrasts.melt(
    id_vars=['subject', 'stim_range'],
    var_name='contrast', value_name='delta_cvr2')

g = sns.catplot(
    data=contrasts_long, x='contrast', y='delta_cvr2',
    col='stim_range', kind='swarm',
    height=4.5, aspect=1.1)
for ax in g.axes.flat:
    ax.axhline(0, color='k', lw=1, ls='--', alpha=0.4)
    ax.tick_params(axis='x', rotation=20)
g.set_axis_labels('Contrast', 'Δ cvR²')
g.set_titles('Range = {col_name}')
g.fig.suptitle('Within-subject cvR² contrasts across the 2×2', y=1.04)
plt.tight_layout()
plt.show()

## Decoding head-to-head

Median absolute error of the posterior-mean numerosity decoded from
FDR-significant voxels in the **test** trials.

In [ ]:
dec_summary = (decoding
               .groupby(['subject', 'pipeline', 'stim_range', 'method'])
               .agg(median_ae=('median_ae', 'mean'),
                    mae=('mae', 'mean'),
                    n_sig=('n_sig_voxels', 'mean'),
                    n_fallback=('fdr_fallback', 'sum'))
               .reset_index())

g = sns.catplot(
    data=dec_summary, x='method', y='median_ae',
    row='pipeline', col='stim_range',
    kind='box', order=METHOD_ORDER, row_order=PIPELINE_ORDER,
    palette=METHOD_COLORS, height=3.6, aspect=1.1)
g.set_axis_labels('Method', 'Median |decoded − true| numerosity')
g.set_titles('Pipeline = {row_name} | Range = {col_name}')
g.fig.suptitle('Decoding error: posterior-mean numerosity vs true',
               y=1.02)
plt.show()

In [ ]:
# Per-trial scatter for one subject as a sanity check — facet rows = pipeline.
subject_of_interest = trials['subject'].iloc[0] if len(trials) else None
if subject_of_interest is not None:
    t = trials[trials['subject'] == subject_of_interest]
    g = sns.relplot(data=t, x='true', y='decoded',
                    row='pipeline', col='stim_range', hue='method',
                    kind='scatter', alpha=0.4,
                    palette=METHOD_COLORS, row_order=PIPELINE_ORDER,
                    height=3.8)
    for ax in g.axes.flat:
        lo = min(t['true'].min(), t['decoded'].min())
        hi = max(t['true'].max(), t['decoded'].max())
        ax.plot([lo, hi], [lo, hi], 'k--', alpha=0.4)
    g.fig.suptitle(f'Decoded vs true — sub-{subject_of_interest}',
                   y=1.03)
    plt.show()

Number of FDR-significant voxels per (method, pipeline) — does the prior
(or smoothing) actually surface more usable voxels?

In [ ]:
g = sns.catplot(
    data=dec_summary, x='method', y='n_sig',
    row='pipeline', col='stim_range',
    kind='box', order=METHOD_ORDER, row_order=PIPELINE_ORDER,
    palette=METHOD_COLORS, height=3.6, aspect=1.1)
g.set_axis_labels('Method', 'Mean # FDR-significant voxels per fold')
g.set_titles('Pipeline = {row_name} | Range = {col_name}')
g.fig.suptitle('Voxel yield at FDR < 0.05 (logit-Gauss mixture)',
               y=1.02)
plt.show()

## GP hyperparameters across subjects

What spatial scale did stage 2 land on? Compare unsmoothed vs smoothed —
smoothing should inflate the apparent lengthscale (the data already looks
smooth before the prior sees it).

In [ ]:
if len(hyperp):
    for metric, ylab in [('lengthscale', 'Lengthscale (mm)'),
                          ('variance',    'GP variance'),
                          ('nugget',      'GP nugget')]:
        g = sns.catplot(
            data=hyperp, x='parameter', y=metric,
            row='pipeline', col='stim_range',
            kind='box', row_order=PIPELINE_ORDER,
            height=3.4, aspect=1.2)
        g.set_axis_labels('Parameter', ylab)
        g.set_titles('Pipeline = {row_name} | Range = {col_name}')
        g.fig.suptitle(f'GP {metric} per regularised parameter', y=1.03)
        plt.show()

## FDR mixture diagnostics

In [ ]:
if 'fdr_r2_threshold' in decoding.columns:
    g = sns.catplot(
        data=decoding, x='method', y='fdr_r2_threshold',
        row='pipeline', col='stim_range', kind='box',
        order=METHOD_ORDER, row_order=PIPELINE_ORDER,
        palette=METHOD_COLORS, height=3.6, aspect=1.1)
    g.set_axis_labels('Method', 'R² threshold at FDR < 0.05')
    g.set_titles('Pipeline = {row_name} | Range = {col_name}')
    g.fig.suptitle('Logit-Gauss mixture FDR R² threshold per fold',
                   y=1.02)
    plt.show()

    fallback_rate = (decoding
                     .groupby(['pipeline', 'stim_range', 'method'])['fdr_fallback']
                     .mean()
                     .unstack('method'))
    print('Fallback rate (proportion of folds where mixture failed and we took top-N):')
    display(fallback_rate)

## Per-subject diagnostic PDFs

Already saved by the fit script — one multi-panel page per (subject, range, pipeline)
showing the R² histogram + GMM components + α=0.05 threshold for every
(fold × method) panel.

```
{pipeline_dir}/sub-{NN}/func/sub-{NN}_range-{narrow,wide}_desc-r2_mixture.pdf
```

In [ ]:
for pipeline, root in PIPELINES.items():
    pdfs = sorted(root.glob('sub-*/func/sub-*_desc-r2_mixture.pdf'))
    print(f'[{pipeline}] {len(pdfs)} diagnostic PDFs')
    for p in pdfs[:3]:
        print('  ', p)